# Fan-out join · **05 — Article images**

The paired bar chart in notebook 04 carries the argument, but it's the
wrong shape for two jobs:

1. **`paired_bars_fan_out_medium.png`** – the same chart redrawn for
   Medium's text column (about 700px wide, so I draw it at 2×, ~1,460px).
   Shorter labels, bigger type, the title split over two lines, so nothing
   shrinks below readable when Medium scales it down.
2. **`social_card.png`** – 1200×630 for Medium's featured image and the
   link preview. All 23 rows can't survive a thumbnail, so I keep six: the
   three treatment functions the fan-out pushes up most and the three it
   pulls down most. That shows the point (two plausible bars, a gap either
   way) at the size people will first see it.

Same palette and layout conventions as my Two Maps article images. Every
number comes from notebook 03's outputs; none are typed in.

In [1]:
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import GOLD_DIR  # noqa: E402

IN = GOLD_DIR / "01_fan_out"
OUT = GOLD_DIR / "02_charts"
OUT.mkdir(parents=True, exist_ok=True)

cmp = pd.read_csv(IN / "fan_out_by_tf.csv")
contrast = pd.read_csv(IN / "contrast.csv").set_index("lookup")
correct_n, fanned_n = contrast.england_pathways.iloc[0], contrast.england_pathways.iloc[1]
correct_pct, fanned_pct = contrast.england_pct_over_18.iloc[0], contrast.england_pct_over_18.iloc[1]

PAPER, INK, INK2, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#9a9891", "#e6e4de"
CORRECT, FANOUT = "#2a78d6", "#eb6834"
plt.rcParams.update({"font.family": "DejaVu Sans", "text.color": INK})


def short(name: str) -> str:
    if name == "Other - Other Services":
        return "Other services"
    return (name.removesuffix(" Service").removesuffix(" Services")
                .replace("Other - ", "Other ").replace(" and ", " & "))


def signed(x: float) -> str:
    return f"{x:+.2f}".replace("-", "−")


def tidy(ax) -> None:
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(axis="y", length=0)
    ax.tick_params(axis="x", length=0, colors=INK2)
    ax.xaxis.grid(True, color=GRID, lw=0.8)
    ax.set_axisbelow(True)


def paired_bars(ax, d: pd.DataFrame, bar_h: float, gap_fs: float, label_fs: float) -> None:
    """Correct grain above, fan-out below, gap in points to the right of each pair."""
    d = d.reset_index(drop=True)
    for i, r in d.iterrows():
        ax.barh(i + bar_h / 2 + 0.02, r.pct_over_18_correct, height=bar_h, color=CORRECT, lw=0)
        ax.barh(i - bar_h / 2 - 0.02, r.pct_over_18_fanout, height=bar_h, color=FANOUT, lw=0)
        ax.text(max(r.pct_over_18_correct, r.pct_over_18_fanout) + 0.9, i, f"{signed(r.gap_points)} pts",
                va="center", ha="left", fontsize=gap_fs, color=INK2)
    ax.set_yticks(range(len(d)), [short(n) for n in d.tf_name], fontsize=label_fs, color=INK)
    ax.set_ylim(-0.65, len(d) - 0.35)
    tidy(ax)

## 1. The chart, at Medium width

In [2]:
d = cmp.sort_values("gap_points", ascending=True)      # largest positive gap at the top
fig, ax = plt.subplots(figsize=(7.3, 10.6), dpi=200)
fig.patch.set_facecolor(PAPER); ax.set_facecolor(PAPER)
paired_bars(ax, d, bar_h=0.36, gap_fs=9, label_fs=9.5)
ax.set_xlim(0, 56)
ax.set_xticks(range(0, 51, 10), [f"{x}%" for x in range(0, 51, 10)], fontsize=9)
ax.set_xlabel("Incomplete pathways waiting over 18 weeks, August 2026", fontsize=9.5, color=INK2)
ax.legend(handles=[Patch(color=CORRECT, label="At pathway grain (matches NHS England)"),
                   Patch(color=FANOUT, label="Through a one-to-many ICB lookup")],
          loc="lower left", bbox_to_anchor=(-0.02, 1.0), frameon=False, fontsize=9.5, handlelength=1.1)
fig.text(0.03, 0.985, "Same file, one extra join.\nNeither bar looks wrong.",
         fontsize=17, fontweight="bold", va="top", ha="left", linespacing=1.15)
fig.text(0.03, 0.915, f"% over 18 weeks by treatment function, England. The join repeats each trust's\n"
         f"rows once per ‘main’ ICB: {correct_n / 1e6:.2f}m pathways become {fanned_n / 1e6:.2f}m, "
         f"and no error is raised.\nGap = fan-out minus correct, percentage points.",
         fontsize=9.5, color=INK2, va="top", ha="left", linespacing=1.4)
fig.text(0.03, 0.008, "Source: NHS England, RTT Waiting Times, August 2026. Reported providers, NONC excluded.",
         fontsize=7.8, color=MUTED, va="bottom")
fig.tight_layout(rect=(0, 0.015, 1, 0.858))
fig.savefig(OUT / "paired_bars_fan_out_medium.png", facecolor=PAPER)
plt.show()

## 2. The social card, 1200×630

Six rows: the three biggest gaps in each direction.

In [3]:
six = pd.concat([cmp.nsmallest(3, "gap_points"), cmp.nlargest(3, "gap_points").iloc[::-1]])
six = six.sort_values("gap_points", ascending=True)

fig = plt.figure(figsize=(12, 6.3), dpi=100)
fig.patch.set_facecolor(PAPER)
ax = fig.add_axes((0.255, 0.115, 0.715, 0.53))
ax.set_facecolor(PAPER)
paired_bars(ax, six, bar_h=0.36, gap_fs=12.5, label_fs=13)
ax.set_xlim(0, 47)
ax.set_xticks(range(0, 41, 10), [f"{x}%" for x in range(0, 41, 10)], fontsize=11.5)

fig.text(0.022, 0.955, "Same file. One extra join.\nNeither number looks wrong.",
         fontsize=33, fontweight="bold", va="top", ha="left", linespacing=1.12)
fig.text(0.022, 0.75, f"% of NHS waiting-list pathways over 18 weeks, August 2026. "
         f"England: {correct_pct:.1f}% correct, {fanned_pct:.1f}% through the fan-out.",
         fontsize=12.5, color=INK2, va="top", ha="left")
# Legend on its own line between the subtitle and the plot, so nothing collides.
ax.legend(handles=[Patch(color=CORRECT, label="correct grain"), Patch(color=FANOUT, label="through the fan-out")],
          loc="lower left", bbox_to_anchor=(-0.01, 1.0), ncol=2, frameon=False, fontsize=12, handlelength=1.1)
fig.text(0.022, 0.035, "NHS England RTT · six of 23 treatment functions, the largest gaps either way",
         fontsize=10.5, color=MUTED, va="bottom")
fig.savefig(OUT / "social_card.png", facecolor=PAPER)
plt.show()

## Alt text for both

In [4]:
top, bottom = cmp.loc[cmp.gap_points.idxmax()], cmp.loc[cmp.gap_points.idxmin()]
card_alt = (
    "Social card: 'Same file. One extra join. Neither number looks wrong.' Paired horizontal bars for six NHS "
    "treatment functions, each showing the percentage of waiting-list pathways over 18 weeks in August 2026 "
    "computed correctly and through a fan-out join. The pairs are nearly the same length; the gaps run from "
    f"{signed(bottom.gap_points)} points ({short(bottom.tf_name)}) to {signed(top.gap_points)} points "
    f"({short(top.tf_name)}). England overall: {correct_pct:.1f}% correct, {fanned_pct:.1f}% through the fan-out."
)
alt_path = OUT / "alt_text.md"
base = alt_path.read_text().split("\n## paired_bars_fan_out_medium.png")[0].rstrip()
medium_alt = base.split("## paired_bars_fan_out.png", 1)[1].strip()
alt_path.write_text(base + "\n\n## paired_bars_fan_out_medium.png\n\nSame description as paired_bars_fan_out.png:\n\n"
                    + medium_alt + "\n\n## social_card.png\n\n" + card_alt + "\n")
print(card_alt)

Social card: 'Same file. One extra join. Neither number looks wrong.' Paired horizontal bars for six NHS treatment functions, each showing the percentage of waiting-list pathways over 18 weeks in August 2026 computed correctly and through a fan-out join. The pairs are nearly the same length; the gaps run from −0.75 points (Dermatology) to +0.69 points (General Internal Medicine). England overall: 34.8% correct, 34.7% through the fan-out.
